In [14]:
# ============================================================
# 03d_rolling_inference.ipynb
# Rolling window inference using trained Transformer
#
# For each stay, for each hour h (4 to stay_length):
#   - Feed last 24h of vitals to Transformer
#   - Read off 12h-ahead risk score
#   - Assign alert tier: no_alert / high_risk / critical
#
# Output: rolling_predictions.csv
#   columns: stay_id, hour, risk_score, alert_tier,
#            true_label_12h, split
# ============================================================

import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('C:/Users/20220505/Desktop/AI-sepsis')
sys.path.append(str(PROJECT_ROOT / 'src'))

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")

# ── Constants ─────────────────────────────────────────────────
MAX_HOURS      = 200
LOOKBACK_W     = 24      # hours of history fed to model
HERO_HORIZON_H = 12      # predict sepsis within next 12h
MIN_HOUR       = 4       # start predicting from hour 4
NUM_BINS       = 48
time_cuts      = np.linspace(0, 200, NUM_BINS + 1)[1:]

ALERT_THRESHOLDS = {
    'no_alert'  : (0.00, 0.50),
    'high_risk' : (0.50, 0.70),
    'critical'  : (0.70, 1.00),
}
ALERT_COLORS = {
    'no_alert'  : '#27AE60',
    'high_risk' : '#E67E22',
    'critical'  : '#C0392B',
}

def get_alert_tier(score):
    if score >= ALERT_THRESHOLDS['critical'][0]:
        return 'critical'
    elif score >= ALERT_THRESHOLDS['high_risk'][0]:
        return 'high_risk'
    else:
        return 'no_alert'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Load metadata ─────────────────────────────────────────────
print("\nLoading metadata...")

with open(str(OUTPUT_DIR / 'rich_feature_names.txt')) as f:
    rich_feature_names = f.read().splitlines()

with open(str(OUTPUT_DIR / 'feature_names.txt')) as f:
    feature_cols = f.read().splitlines()

stay_ids_order = pd.read_csv(
    str(OUTPUT_DIR / 'stay_ids_order.csv')
).squeeze().tolist()
stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}

print(f"  Rich features : {len(rich_feature_names)}")
print(f"  Static features: {len(feature_cols)}")
print(f"  Stays         : {len(stay_ids_order):,}")

# ── Load X_rich_full ──────────────────────────────────────────
print("\nLoading X_rich_full.npy (1.49 GB)...")
X_rich_full = np.load(str(OUTPUT_DIR / 'X_rich_full.npy'))
print(f"  Shape : {X_rich_full.shape}")

# ── Load static features ──────────────────────────────────────
print("Loading engineered features...")
all_features  = pd.read_csv(str(OUTPUT_DIR / 'engineered_features.csv'))
static_lookup = (
    all_features
    .set_index('stay_id')[feature_cols]
    .fillna(0)
)
print(f"  Shape : {all_features.shape}")

# ── Load hourly labels ────────────────────────────────────────
print("Loading hourly labels...")
hl = pd.read_csv(str(OUTPUT_DIR / 'hourly_labels.csv'))
print(f"  Shape : {hl.shape}")

# ── Load split info ───────────────────────────────────────────
print("Loading splits...")
split_df  = pd.read_csv(str(OUTPUT_DIR / 'subject_splits.csv'))
cohort    = pd.read_csv(
    str(OUTPUT_DIR / 'icu_cohort (1).csv'),
    usecols=['stay_id', 'subject_id']
)
stay_split = (
    cohort
    .merge(split_df, on='subject_id', how='left')
    .set_index('stay_id')['split']
    .to_dict()
)
print(f"  Split counts: "
      f"{pd.Series(stay_split).value_counts().to_dict()}")

print("\nSetup complete ✓")

Device : cuda

Loading metadata...
  Rich features : 25
  Static features: 127
  Stays         : 74,550

Loading X_rich_full.npy (1.49 GB)...
  Shape : (74550, 200, 25)
Loading engineered features...
  Shape : (89075, 128)
Loading hourly labels...
  Shape : (2618839, 8)
Loading splits...
  Split counts: {'train': 58675, 'test': 12553, 'val': 12509}

Setup complete ✓


In [15]:
# ============================================================
# Cell 2: Load RollingTransformer architecture and weights
# Must match exactly what was defined in 03b_survival_modeling12_rolling
# ============================================================

class RollingTransformer(nn.Module):
    def __init__(self,
                 vital_dim    = 25,
                 static_dim   = 127,
                 d_model      = 128,
                 nhead        = 4,
                 n_layers     = 2,
                 static_hidden= 64,
                 fusion_hidden= 128,
                 dropout      = 0.2,
                 max_seq_len  = 25):
        super().__init__()
        self.d_model = d_model

        self.vital_proj = nn.Linear(vital_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_emb    = nn.Embedding(max_seq_len + 1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward= d_model * 4,
            dropout        = dropout,
            batch_first    = True,
            norm_first     = True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)

        self.static_enc = nn.Sequential(
            nn.Linear(static_dim, static_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(d_model + static_hidden),
            nn.Linear(d_model + static_hidden, fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),   # single sigmoid output
        )

    def forward(self, x_seq, x_static, lengths):
        B, T, _ = x_seq.shape
        x   = self.vital_proj(x_seq)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        pos = torch.arange(T + 1, device=x.device).unsqueeze(0)
        x   = x + self.pos_emb(pos)

        mask = torch.ones(B, T + 1, dtype=torch.bool, device=x.device)
        mask[:, 0] = False
        for i, l in enumerate(lengths):
            mask[i, 1:l.item() + 1] = False

        out     = self.transformer(x, src_key_padding_mask=mask)
        cls_out = out[:, 0, :]
        s_out   = self.static_enc(x_static)
        fused   = torch.cat([cls_out, s_out], dim=1)
        return torch.sigmoid(self.fusion(fused)).squeeze(1)


# ── Instantiate and load weights ──────────────────────────────
model = RollingTransformer(
    vital_dim    = len(rich_feature_names),
    static_dim   = len(feature_cols),
    d_model      = 128,
    nhead        = 4,
    n_layers     = 2,
    static_hidden= 64,
    fusion_hidden= 128,
    dropout      = 0.2,
    max_seq_len  = 25,
).to(device)

model.load_state_dict(torch.load(
    str(OUTPUT_DIR / 'rolling_transformer_best.pt'),
    map_location=device,
    weights_only=True,
))
model.eval()

total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Model loaded : {total_params:,} parameters")
print(f"Device       : {device}")
print("Output: sigmoid P(sepsis within 12h) — no conversion needed ✓")

Model loaded : 436,737 parameters
Device       : cuda
Output: sigmoid P(sepsis within 12h) — no conversion needed ✓


In [16]:
# ============================================================
# Cell 3: Build rolling 12h-ahead labels from hourly_labels
#
# For each stay-hour (stay_id, h):
#   y_roll = 1 if sepsis onset occurs between h and h+12
#          = 0 otherwise
# ============================================================
print("Building rolling 12h-ahead labels...")

# ── Find sepsis onset hour per stay ───────────────────────────
sepsis_onset = (
    hl[hl['label'] == 1]
    .groupby('stay_id')['hour']
    .min()
    .reset_index()
    .rename(columns={'hour': 'onset_hour'})
)
onset_lookup = sepsis_onset.set_index('stay_id')['onset_hour'].to_dict()

# ── Max observed hour per stay ────────────────────────────────
max_hour_per_stay = (
    hl.groupby('stay_id')['hour']
    .max()
    .to_dict()
)

print(f"Stays with sepsis onset : {len(onset_lookup):,}")
print(f"Total stays             : {len(stay_ids_order):,}")

# ── Build label lookup: (stay_id, hour) → y_roll ─────────────
# We will use this during inference to assign labels
def get_rolling_label(stay_id, hour, horizon=HERO_HORIZON_H):
    """
    Returns 1 if sepsis onset occurs between hour and hour+horizon.
    Returns 0 if censored or onset outside window.
    Returns -1 if hour exceeds stay length (should not be evaluated).
    """
    max_h = max_hour_per_stay.get(stay_id, 0)
    if hour > max_h:
        return -1   # beyond stay — skip

    onset = onset_lookup.get(stay_id, None)
    if onset is None:
        return 0    # no sepsis — censored

    if hour <= onset < hour + horizon:
        return 1    # sepsis onset within next horizon hours
    else:
        return 0    # sepsis outside window or already occurred

# ── Quick verification ────────────────────────────────────────
print("\nLabel verification (3 sepsis stays):")
for sid in list(onset_lookup.keys())[:3]:
    onset = onset_lookup[sid]
    for h in [onset - 6, onset - 1, onset, onset + 1]:
        if h >= 0:
            lbl = get_rolling_label(sid, h)
            print(f"  stay={sid} hour={h:>4} onset={onset:>4} "
                  f"→ label={lbl}")

print("\nRolling labels ready ✓")

Building rolling 12h-ahead labels...
Stays with sepsis onset : 26,643
Total stays             : 74,550

Label verification (3 sepsis stays):
  stay=30000646 hour=   0 onset=   1 → label=1
  stay=30000646 hour=   1 onset=   1 → label=1
  stay=30000646 hour=   2 onset=   1 → label=0
  stay=30001148 hour=   0 onset=   1 → label=1
  stay=30001148 hour=   1 onset=   1 → label=1
  stay=30001148 hour=   2 onset=   1 → label=0
  stay=30001396 hour=   4 onset=  10 → label=1
  stay=30001396 hour=   9 onset=  10 → label=1
  stay=30001396 hour=  10 onset=  10 → label=1
  stay=30001396 hour=  11 onset=  10 → label=0

Rolling labels ready ✓


In [17]:
# ============================================================
# Cell 4: Rolling inference — main loop
#
# Processes all stays in batches for efficiency.
# For each stay, slides hour by hour through X_rich_full
# using a 24h lookback window.
# Expected time: 20-40 minutes for 74,550 stays
# ============================================================
import time

print("Starting rolling inference...")
print(f"  Lookback window : {LOOKBACK_W}h")
print(f"  Prediction horizon: {HERO_HORIZON_H}h")
print(f"  Min hour        : {MIN_HOUR}")
print(f"  Max hours       : {MAX_HOURS}")
print(f"  Alert thresholds: {ALERT_THRESHOLDS}")
print()

BATCH_SIZE = 256
results    = []
t_start    = time.time()

# ── Pre-load static features as tensor ───────────────────────
print("Pre-loading static features...")
X_static_all = np.zeros(
    (len(stay_ids_order), len(feature_cols)),
    dtype=np.float32
)
for i, sid in enumerate(stay_ids_order):
    if sid in static_lookup.index:
        X_static_all[i] = static_lookup.loc[sid].values.astype(np.float32)

X_static_tensor = torch.tensor(X_static_all, dtype=torch.float32)
print(f"Static features shape : {X_static_all.shape}")

# ── Build list of all (stay_idx, hour) pairs to evaluate ──────
print("Building evaluation schedule...")
eval_pairs = []

for i, sid in enumerate(stay_ids_order):
    max_h = min(
        max_hour_per_stay.get(sid, MIN_HOUR),
        MAX_HOURS - 1
    )
    for h in range(MIN_HOUR, max_h + 1):
        eval_pairs.append((i, sid, h))

total_pairs = len(eval_pairs)
print(f"Total stay-hours to evaluate : {total_pairs:,}")
print(f"Estimated time @ {BATCH_SIZE} batch: "
      f"~{total_pairs // BATCH_SIZE // 60 + 1} min\n")

# ── Batch inference ───────────────────────────────────────────
model.eval()

for batch_start in range(0, total_pairs, BATCH_SIZE):
    batch = eval_pairs[batch_start:batch_start + BATCH_SIZE]

    x_seqs    = []
    x_statics = []
    lengths   = []
    meta      = []

    for stay_idx, stay_id, hour in batch:
        h_start = max(0, hour - LOOKBACK_W)
        seq     = X_rich_full[stay_idx, h_start:hour, :]
        seq_len = len(seq)

        if seq_len == 0:
            continue

        x_seqs.append(torch.tensor(seq, dtype=torch.float32))
        x_statics.append(X_static_tensor[stay_idx])
        lengths.append(seq_len)
        meta.append((stay_id, hour, stay_split.get(stay_id, 'unknown')))

    if not x_seqs:
        continue

    # Pad sequences to same length
    max_len  = max(lengths)
    x_padded = torch.zeros(len(x_seqs), max_len, len(rich_feature_names))
    for i, seq in enumerate(x_seqs):
        x_padded[i, :len(seq), :] = seq

    x_padded   = x_padded.to(device)
    x_static_b = torch.stack(x_statics).to(device)
    lengths_t  = torch.tensor(lengths, dtype=torch.long).to(device)

    # Inference — direct sigmoid output, no conversion needed
    with torch.no_grad():
        risk_scores = model(x_padded, x_static_b, lengths_t).cpu().numpy()

    for i, (sid, hour, split) in enumerate(meta):
        risk_12h = float(np.clip(risk_scores[i], 0.0, 1.0))
        tier     = get_alert_tier(risk_12h)
        label    = get_rolling_label(sid, hour)

        if label == -1:
            continue

        results.append({
            'stay_id'       : sid,
            'hour'          : hour,
            'risk_score'    : round(risk_12h, 4),
            'alert_tier'    : tier,
            'true_label_12h': label,
            'split'         : split,
        })

    # Progress
    if (batch_start // BATCH_SIZE) % 500 == 0:
        elapsed = time.time() - t_start
        pct     = batch_start / total_pairs
        eta     = (elapsed / max(pct, 1e-6)) * (1 - pct) / 60
        print(f"  {batch_start:>8,} / {total_pairs:,} "
              f"({pct:.1%}) | "
              f"elapsed={elapsed/60:.1f}min | "
              f"ETA={eta:.1f}min",
              end='\r')

print(f"\n\nInference complete.")
print(f"Total time : {(time.time()-t_start)/60:.1f} minutes")
print(f"Results    : {len(results):,} stay-hour predictions")

# ── Save ──────────────────────────────────────────────────────
rolling_df = pd.DataFrame(results)
rolling_df.to_csv(
    str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False
)
print(f"\nSaved → rolling_predictions.csv ✓")
print(f"Shape  : {rolling_df.shape}")
print(f"\nPreview:")
print(rolling_df.head(10).to_string(index=False))

Starting rolling inference...
  Lookback window : 24h
  Prediction horizon: 12h
  Min hour        : 4
  Max hours       : 200
  Alert thresholds: {'no_alert': (0.0, 0.5), 'high_risk': (0.5, 0.7), 'critical': (0.7, 1.0)}

Pre-loading static features...
Static features shape : (74550, 127)
Building evaluation schedule...
Total stay-hours to evaluate : 2,296,449
Estimated time @ 256 batch: ~150 min

  2,176,000 / 2,296,449 (94.8%) | elapsed=4.5min | ETA=0.3minin

Inference complete.
Total time : 4.8 minutes
Results    : 2,296,449 stay-hour predictions

Saved → rolling_predictions.csv ✓
Shape  : (2296449, 6)

Preview:
 stay_id  hour  risk_score alert_tier  true_label_12h split
30000153     4      0.7126   critical               0 train
30000153     5      0.6873  high_risk               0 train
30000153     6      0.6606  high_risk               0 train
30000153     7      0.6367  high_risk               0 train
30000153     8      0.6140  high_risk               0 train
30000153     9    

In [24]:
# ============================================================
# Cell 5: Validate rolling predictions
# ============================================================
print("Validating rolling predictions...")
print(f"\nShape : {rolling_df.shape}")

# ── Distribution of alert tiers ───────────────────────────────
print("\nAlert tier distribution:")
tier_counts = rolling_df['alert_tier'].value_counts()
for tier, count in tier_counts.items():
    pct = count / len(rolling_df)
    print(f"  {tier:<12} : {count:>8,}  ({pct:.2%})")

# ── Risk score distribution ───────────────────────────────────
print(f"\nRisk score stats:")
print(f"  Mean   : {rolling_df['risk_score'].mean():.4f}")
print(f"  Median : {rolling_df['risk_score'].median():.4f}")
print(f"  Std    : {rolling_df['risk_score'].std():.4f}")
print(f"  Min    : {rolling_df['risk_score'].min():.4f}")
print(f"  Max    : {rolling_df['risk_score'].max():.4f}")

# ── Label distribution ────────────────────────────────────────
print(f"\nLabel distribution:")
lbl_counts = rolling_df['true_label_12h'].value_counts()
print(f"  Positive (sepsis within 12h) : "
      f"{lbl_counts.get(1,0):>8,} "
      f"({lbl_counts.get(1,0)/len(rolling_df):.3%})")
print(f"  Negative                     : "
      f"{lbl_counts.get(0,0):>8,} "
      f"({lbl_counts.get(0,0)/len(rolling_df):.3%})")

# ── AUROC on test set ─────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

test_df = rolling_df[rolling_df['split'] == 'test']
print(f"\nTest set rolling evaluation:")
print(f"  Stay-hours : {len(test_df):,}")

auroc = roc_auc_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
auprc = average_precision_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")

# ── Per-split breakdown ───────────────────────────────────────
print(f"\nPer-split breakdown:")
for split in ['train', 'val', 'test']:
    sub = rolling_df[rolling_df['split'] == split]
    if len(sub) == 0:
        continue
    pos_rate = sub['true_label_12h'].mean()
    print(f"  {split:<6} : {len(sub):>8,} stay-hours | "
          f"positive rate={pos_rate:.3%}")

# ── Sample patient timeline ───────────────────────────────────
print(f"\nSample patient timeline (first sepsis patient):")
sepsis_stays = rolling_df[
    rolling_df['true_label_12h'] == 1
]['stay_id'].unique()

if len(sepsis_stays) > 0:
    sample_sid = sepsis_stays[0]
    sample     = rolling_df[
        rolling_df['stay_id'] == sample_sid
    ].sort_values('hour')
    onset      = onset_lookup.get(sample_sid, '?')
    print(f"  stay_id={sample_sid}  onset={onset}h")
    print(f"  {'Hour':>5} {'Risk':>8} {'Tier':<12} {'Label':>6}")
    print(f"  {'-'*35}")
    for _, row in sample.iterrows():
        marker = ' ← ONSET' if row['hour'] == onset else ''
        print(f"  {row['hour']:>5.0f} "
              f"{row['risk_score']:>8.4f} "
              f"{row['alert_tier']:<12} "
              f"{row['true_label_12h']:>6}{marker}")

print("\n03d complete ✓")
print("Ready for updated 04 — rolling evaluation")

Validating rolling predictions...

Shape : (2296449, 6)

Alert tier distribution:
  no_alert     : 1,968,728  (85.73%)
  high_risk    :  238,355  (10.38%)
  critical     :   89,366  (3.89%)

Risk score stats:
  Mean   : 0.2760
  Median : 0.2346
  Std    : 0.1946
  Min    : 0.0024
  Max    : 0.9834

Label distribution:
  Positive (sepsis within 12h) :   36,020 (1.569%)
  Negative                     : 2,260,429 (98.431%)

Test set rolling evaluation:
  Stay-hours : 350,688
  AUROC      : 0.7558
  AUPRC      : 0.0771

Per-split breakdown:
  train  : 1,596,043 stay-hours | positive rate=1.599%
  val    :  349,718 stay-hours | positive rate=1.500%
  test   :  350,688 stay-hours | positive rate=1.496%

Sample patient timeline (first sepsis patient):
  stay_id=30001396  onset=10h
   Hour     Risk Tier          Label
  -----------------------------------
      4   0.8407 critical          1
      5   0.8215 critical          1
      6   0.8026 critical          1
      7   0.7841 critical    

In [25]:
# ============================================================
# Diagnostic: AUROC by hour range
# Tells us where in the stay the model performs well
# ============================================================
from sklearn.metrics import roc_auc_score

test_roll = rolling_df[rolling_df['split'] == 'test'].copy()

print("AUROC by hour range — test set")
print(f"{'Hour range':>15} {'Stay-hours':>12} "
      f"{'Positives':>10} {'AUROC':>8}")
print("-" * 50)

bins = [(4,12), (12,24), (24,48), (48,96), (96,200)]

for lo, hi in bins:
    sub = test_roll[
        (test_roll['hour'] >= lo) &
        (test_roll['hour'] <  hi)
    ]
    n_pos = sub['true_label_12h'].sum()
    if n_pos < 10 or (len(sub) - n_pos) < 10:
        print(f"  {lo:>3}h – {hi:>3}h : insufficient events")
        continue
    try:
        auroc = roc_auc_score(
            sub['true_label_12h'], sub['risk_score']
        )
        print(f"  {lo:>3}h – {hi:>3}h : "
              f"{len(sub):>10,}  "
              f"{n_pos:>10,}  "
              f"{auroc:>8.4f}")
    except Exception as e:
        print(f"  {lo:>3}h – {hi:>3}h : error — {e}")

# Also check AUROC on only hours 4-24 (in-distribution)
sub_early = test_roll[test_roll['hour'] < 24]
auroc_early = roc_auc_score(
    sub_early['true_label_12h'], sub_early['risk_score']
)
print(f"\nEarly hours only (4-24h, in-distribution): "
      f"AUROC={auroc_early:.4f}")
print(f"All hours (4-200h):                        "
      f"AUROC={test_roll['true_label_12h'].pipe(lambda y: roc_auc_score(y, test_roll['risk_score'])):.4f}")

AUROC by hour range — test set
     Hour range   Stay-hours  Positives    AUROC
--------------------------------------------------
    4h –  12h :     64,178       2,220    0.7778
   12h –  24h :     76,677         890    0.7159
   24h –  48h :     91,875         893    0.6835
   48h –  96h :     75,437         755    0.6808
   96h – 200h :     42,521         490    0.7062

Early hours only (4-24h, in-distribution): AUROC=0.7873
All hours (4-200h):                        AUROC=0.7558


## Threshold Calibration

In [26]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, f1_score

fpr, tpr, threshs = roc_curve(test_roll['true_label_12h'], test_roll['risk_score'])
youden_idx  = np.argmax(tpr - fpr)
best_thresh = threshs[youden_idx]

y_pred = (test_roll['risk_score'] >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(test_roll['true_label_12h'], y_pred).ravel()

print(f"Youden-optimal threshold : {best_thresh:.4f}")
print(f"  Sensitivity : {tp/(tp+fn):.4f}")
print(f"  Specificity : {tn/(tn+fp):.4f}")
print(f"  PPV         : {tp/(tp+fp):.4f}")
print(f"  NPV         : {tn/(tn+fn):.4f}")
print(f"  F1          : {f1_score(test_roll['true_label_12h'], y_pred):.4f}")
print(f"  Alert rate  : {y_pred.mean():.3%}")

Youden-optimal threshold : 0.3832
  Sensitivity : 0.6477
  Specificity : 0.7427
  PPV         : 0.0368
  NPV         : 0.9928
  F1          : 0.0697
  Alert rate  : 26.312%


## Lead time analysis

In [27]:
lead_times = []
for sid in test_roll[test_roll['true_label_12h'] == 1]['stay_id'].unique():
    onset  = onset_lookup.get(sid)
    if onset is None:
        continue
    stay   = test_roll[test_roll['stay_id'] == sid].sort_values('hour')
    alerts = stay[stay['risk_score'] >= best_thresh]['hour']
    if len(alerts) > 0:
        first_alert = alerts.min()
        lead_time   = onset - first_alert
        if lead_time >= 0:
            lead_times.append(lead_time)

lead_times = np.array(lead_times)
print(f"Lead time analysis (hours before onset):")
print(f"  Patients with early alert : {len(lead_times):,}")
print(f"  Mean lead time  : {lead_times.mean():.1f}h")
print(f"  Median lead time: {np.median(lead_times):.1f}h")
print(f"  ≥6h lead time   : {(lead_times >= 6).mean():.1%}")
print(f"  ≥12h lead time  : {(lead_times >= 12).mean():.1%}")

Lead time analysis (hours before onset):
  Patients with early alert : 662
  Mean lead time  : 22.2h
  Median lead time: 7.0h
  ≥6h lead time   : 57.3%
  ≥12h lead time  : 36.9%


## Threshold optimization

In [28]:
from sklearn.metrics import confusion_matrix

print(f"{'Thresh':>8} {'Sens':>7} {'Spec':>7} {'PPV':>7} {'NPV':>7} {'Alert%':>8} {'F1':>7}")
print("-" * 60)

for thresh in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    y_pred = (test_roll['risk_score'] >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_roll['true_label_12h'], y_pred).ravel()
    sens  = tp / (tp + fn)
    spec  = tn / (tn + fp)
    ppv   = tp / (tp + fp + 1e-9)
    npv   = tn / (tn + fn + 1e-9)
    f1    = 2*tp / (2*tp + fp + fn + 1e-9)
    alert = y_pred.mean()
    print(f"  {thresh:.2f}   {sens:>7.3f} {spec:>7.3f} {ppv:>7.3f} {npv:>7.3f} {alert:>7.2%} {f1:>7.4f}")

  Thresh    Sens    Spec     PPV     NPV   Alert%      F1
------------------------------------------------------------
  0.20     0.857   0.417   0.022   0.995  58.67%  0.0426
  0.30     0.745   0.616   0.029   0.994  38.90%  0.0552
  0.40     0.624   0.764   0.039   0.993  24.14%  0.0728
  0.50     0.479   0.859   0.049   0.991  14.59%  0.0891
  0.60     0.382   0.923   0.070   0.990   8.17%  0.1184
  0.70     0.273   0.965   0.105   0.989   3.89%  0.1520
  0.80     0.137   0.989   0.160   0.987   1.28%  0.1476


In [29]:
# Faster — just reclassify existing predictions without rerunning inference
rolling_df['alert_tier'] = rolling_df['risk_score'].apply(get_alert_tier)
rolling_df.to_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False)
print("Alert tiers updated and saved ✓")
print(rolling_df['alert_tier'].value_counts())

Alert tiers updated and saved ✓
alert_tier
no_alert     1968656
high_risk     238404
critical       89389
Name: count, dtype: int64


In [30]:
# Reclassify existing predictions with updated thresholds
def get_alert_tier(score):
    if score >= 0.70: return 'critical'
    elif score >= 0.50: return 'high_risk'
    else: return 'no_alert'

rolling_df['alert_tier'] = rolling_df['risk_score'].apply(get_alert_tier)
rolling_df.to_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False)
print("Alert tiers updated and saved ✓")
print(rolling_df['alert_tier'].value_counts())

Alert tiers updated and saved ✓
alert_tier
no_alert     1968656
high_risk     238404
critical       89389
Name: count, dtype: int64


## Save final results

In [31]:
import json
results_summary = {
    'model'      : 'RollingTransformer',
    'auroc'      : round(float(roc_auc_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'auprc'      : round(float(average_precision_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'best_thresh': round(float(best_thresh), 4),
    'sensitivity': round(float(tp/(tp+fn)), 4),
    'specificity': round(float(tn/(tn+fp)), 4),
    'ppv'        : round(float(tp/(tp+fp)), 4),
    'npv'        : round(float(tn/(tn+fn)), 4),
    'f1'         : round(float(f1_score(test_roll['true_label_12h'], y_pred)), 4),
    'alert_rate' : round(float(y_pred.mean()), 4),
    'mean_lead_time_h'  : round(float(lead_times.mean()), 1),
    'median_lead_time_h': round(float(np.median(lead_times)), 1),
    'pct_ge_6h_lead'    : round(float((lead_times >= 6).mean()), 4),
    'pct_ge_12h_lead'   : round(float((lead_times >= 12).mean()), 4),
}
with open(str(OUTPUT_DIR / 'rolling_results.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)
print("Saved → rolling_results.json ✓")
print("\n03d complete ✓")
print("Ready for 04_results_and_evaluation")

Saved → rolling_results.json ✓

03d complete ✓
Ready for 04_results_and_evaluation
